# Observer.ai: fine-tuning Laya on labeled agent traces

Fine-tunes a released **Laya** checkpoint on your own labeled traces, then checks whether it beats the zero-shot model and a no-model heuristic as a sampler. It follows `observer_kev_lora_finetune.ipynb` step for step, with the same data, split, questions and metrics, so the two runs can be compared later. This notebook does not compare them.

Laya is an encoder (ModernBERT family) with a small decision head. It answers the same typed questions as Kev (`noul`, `score`, `choice`) in one forward pass and speaks the same `/v1/systemone` protocol. Two differences shape this notebook:
- **Full fine-tune, not LoRA.** `laya-train` updates the encoder and the head.
- **Long slices are truncated, not dropped.** Laya reads a fixed token window and cuts the end of a state that does not fit. The final answer is at the end of our slice, so the window is sized to fit every slice.

**Flow**
1. Load labeled traces and build trace slices (the `state`)
2. Convert to Laya's training JSONL and split by question: train / calibration / test
3. Zero-shot baseline: score the test portion with the released checkpoint
4. Size the token window, then fine-tune with `laya-train`
5. Score calibration and test with the fine-tuned run, for both signals: `p(answer_wrong)` and `p(keep)`
6. Fit a temperature per signal on calibration, report ECE on test
7. Heuristic baseline (Arm D) and the success bar
8. One results table at realistic base rates, and a model manifest

In [18]:
import hashlib, json, math, os, random, re, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---- config ----
TRACES_PATH  = Path("data/eval/labeled_traces.jsonl")   # your labeled traces (schema below)
WORK_DIR     = Path("runs/observer-laya-ft").resolve()
LAYA_PATH    = Path("/Users/samuelabolo/kode/hobby/laya")   # clone of github.com/NandhaKishorM/laya
BASE_MODEL   = "convaiinnovations/laya-multilingual"    # 322M, mmBERT-base. Or convaiinnovations/laya (421M, English)
RUN_DIR      = LAYA_PATH / "runs" / "observer-laya-ft"  # where the fine-tuned checkpoint is written
DEVICE       = "mps"                                # Apple Silicon; cuda or cpu elsewhere
SAMPLE_RATE  = 0.05                                 # the headline budget (success bar)
SAMPLE_RATES = [0.05, 0.10, 0.20]                   # every budget in the results table
TARGET_BASE_RATES = [0.10, 0.20]                    # share of wrong traces in the rebalanced test sets
N_RESAMPLES  = 20                                   # rebalanced test sets per base rate (seeds SEED, SEED+1, ...)
HELDOUT_FRAC = 0.2                                  # only used when the dataset has no `split` field
CALIB_FRAC   = 0.5                                  # share of held-out questions used to fit temperatures
SEED         = 7

DECISION_SIGNAL = "keep_or_drop"                    # the question the engine decides on: keep_or_drop | answer_wrong
SUCCESS_MARGIN  = 0.10                              # fine-tuned must beat the best heuristic by this much normalized recall

# token window. Laya cuts the end of a state that does not fit, and the final answer is at the end.
MAX_LEN            = None                           # None = smallest multiple of 128 that fits every slice
MAX_TRUNCATED_FRAC = 0.0                            # fail if more slices than this would be cut

# training. One optimizer update covers TRAIN_BATCH * TRAIN_ACCUM = 64 items; an item is one question on one trace.
# Memory grows with TRAIN_BATCH x MAX_LEN^2. Batch 8 without checkpointing took 40 GB on MPS and thrashed swap;
# batch 2 with checkpointing stayed under 4 GB and ran at about 0.2 s per item.
TRAIN_EPOCHS = 2
TRAIN_BATCH  = 2                                    # items per forward pass
TRAIN_ACCUM  = 32                                   # forward passes per optimizer update
TRAIN_CHECKPOINTING = True                          # recompute activations in the backward pass instead of storing them
ENCODER_LR   = 2.5e-5                               # laya-train defaults
HEAD_LR      = 1e-4
TRAIN_LOSS   = "rlcd"                               # rlcd | soft-ce

# Each server registers one checkpoint under a name, and every request names it.
ZERO_SHOT_URL,  ZERO_SHOT_MODEL  = "http://127.0.0.1:8011/v1/systemone", "observer-base"   # released checkpoint
FINETUNED_URL,  FINETUNED_MODEL  = "http://127.0.0.1:8012/v1/systemone", "observer-ft"     # your fine-tuned run

random.seed(SEED)
np.random.seed(SEED)
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load labeled traces

Expected schema, one JSON object per line (produced by your ReAct agent + gold-answer labeller):

```json
{
  "trace_id": "7f3a91c2e8",
  "question": "Which river flows through the capital of France?",
  "tool_calls": [{"name": "search", "args": "capital of France", "status": "ok"}],
  "evidence": ["The Eiffel Tower is in Paris, France."],
  "final_answer": "The Rhône flows through the capital, Lyon.",
  "label": "wrong"
}
```

`label` is one of `correct`, `partial`, `wrong`.

In [19]:
def load_traces(path):
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    bad = [r["trace_id"] for r in rows if r["label"] not in {"correct", "partial", "wrong"}]
    assert not bad, f"unknown labels on {len(bad)} traces, e.g. {bad[:3]}"
    return rows

traces = load_traces(TRACES_PATH)
labels = [t["label"] for t in traces]
print(f"{len(traces)} traces:", {k: labels.count(k) for k in ("correct", "partial", "wrong")})

4499 traces: {'correct': 1729, 'partial': 625, 'wrong': 2145}


## 2. Build trace slices

The `state` is the slice the Go engine will send: question, tool calls, top evidence and the final answer.

`build_state` is identical to the Kev notebook's and to the Go engine's `BuildState`, so every model is shown the same text.

In [20]:
MAX_EVIDENCE = 3
MAX_EVIDENCE_CHARS = 1500

def build_state(t):
    tools = "\n".join(
        f"{i}. {c['name']}({c.get('args', '')!r}) -> {c.get('status', 'ok')}"
        for i, c in enumerate(t.get("tool_calls", []), 1)
    ) or "(none)"
    evidence = "\n".join(
        f"[{i}] {e[:MAX_EVIDENCE_CHARS]}" for i, e in enumerate(t.get("evidence", [])[:MAX_EVIDENCE], 1)
    ) or "(none)"
    return (
        f"USER QUESTION:\n{t['question']}\n\n"
        f"TOOL CALLS:\n{tools}\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"FINAL ANSWER:\n{t['final_answer']}"
    )

print(build_state(traces[0]))

USER QUESTION:
What was the estimated population of the city that Luis Villalta Aquino was from in 2012?

TOOL CALLS:
1. search({'query': 'Luis Villalta Aquino city 2012 population'}) -> ok

EVIDENCE:
[1] Luis Villalta Aquino (October 2, 1969, Lima, Peru – March 3, 2004, Pompano Beach, Florida) was a professional boxer, who was nicknamed "El Puma" during his career.
[2] San Luis Potosí, commonly called SLP or simply San Luis, is the capital and the most populous city of the Mexican state of San Luis Potosí. The city lies at an elevation of 1850 m . It has an estimated population of 735,886 in the city proper and a population of approximately 1,021,688 in its metropolitan area, formed with the neighbour city of Soledad de Graciano Sánchez and some other small townships inside the urban area, which makes the metropolitan area of Greater San Luis Potosí the eleventh largest in Mexico.
[3] Pompano Beach is a city in Broward County, Florida, United States, along the coast of the Atlantic Oc

### Laya's environment and tokenizer

Laya runs in its own environment inside `LAYA_PATH` (`uv run` creates it on first use), the same way Kev does. This cell downloads the base checkpoint once and asks Laya itself how many tokens each slice takes, so the counts are the ones training and serving will see.

In [21]:
# uv warns when VIRTUAL_ENV points at this notebook's environment instead of Laya's
LAYA_ENV = {k: v for k, v in os.environ.items() if k != "VIRTUAL_ENV"}
LAYA_ENV["PYTHONUNBUFFERED"] = "1"

def laya_python(code, *args, stdin=None):
    """Run a Python snippet inside Laya's environment and return the JSON it prints last."""
    done = subprocess.run(
        ["uv", "run", "--extra", "serve", "python", "-c", code, *map(str, args)],
        cwd=LAYA_PATH, env=LAYA_ENV, input=stdin, capture_output=True, text=True,
    )
    if done.returncode:
        raise RuntimeError(f"Laya's environment failed:\n{done.stderr[-2000:]}")
    return json.loads(done.stdout.strip().splitlines()[-1])

COUNT_TOKENS = """
import json, sys
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.train import encode_state, resolve_checkpoint_dir

base_dir = resolve_checkpoint_dir(sys.argv[1])
_fix_tokenizer_config(base_dir)
tok = AutoTokenizer.from_pretrained(base_dir + "/tokenizer")
cfg = json.load(open(base_dir + "/rl_agent_config.json"))
states = [json.loads(line) for line in sys.stdin if line.strip()]
print(json.dumps({
    "base_dir": base_dir,
    "max_len": cfg.get("max_len", 512), "head_max_len": cfg.get("head_max_len", 192),
    "state_tokens": [len(encode_state(tok, s, 10**9)) for s in states],
}))
"""

info = laya_python(COUNT_TOKENS, BASE_MODEL, stdin="\n".join(json.dumps(build_state(t)) for t in traces))
BASE_DIR = Path(info["base_dir"])
BASE_MAX_LEN, HEAD_MAX_LEN = info["max_len"], info["head_max_len"]
lengths = np.array(info["state_tokens"])
print(f"base checkpoint: {BASE_DIR}")
print(f"its default window: {BASE_MAX_LEN} tokens in total, of which up to {HEAD_MAX_LEN} for the question")
print(f"tokens per slice: median {int(np.median(lengths))}, p95 {int(np.percentile(lengths, 95))}, max {lengths.max()}")

base checkpoint: /Users/samuelabolo/.cache/huggingface/hub/models--convaiinnovations--laya-multilingual/snapshots/1720e3e3357cfe1e281542e223f8273b0890ca34
its default window: 1024 tokens in total, of which up to 256 for the question
tokens per slice: median 381, p95 615, max 995


## 3. Questions and Laya training format

The questions are the Kev notebook's, word for word. Laya's training file keeps them apart from the answers: each row is `{state, questions, expected}`, where `expected` holds `true`/`false` for a `noul` and the option name for a `choice`.

**Two decision signals**, both trained and both evaluated. Partial answers count as wrong in each, matching the main metric.
- `answer_wrong` (noul): `p(answer_wrong)`
- `keep_or_drop` (choice): `p(keep)`, the probability of the `keep` option

`DECISION_SIGNAL` picks the one the Go engine decides on.

**Split:** by question, never by trace, so natural and seeded runs of one question stay on the same side.

Laya gives each option description at most 48 tokens. The two `keep_or_drop` descriptions fit.

In [22]:
QUESTIONS = {
    "answer_wrong": {
        "type": "noul",
        "instructions": "Is the final answer wrong or only partially correct for the user's question?",
    },
    "keep_or_drop": {
        "type": "choice",
        "instructions": "Should this agent trace be kept for human review?",
        "criteria": {
            "keep": "The final answer is wrong, only partly correct or unsupported by the evidence, or the agent misused or skipped its tools.",
            "drop": "The final answer is correct, complete and supported by the evidence, and the agent used its tools appropriately.",
        },
    },
}

def to_laya_record(t):
    expected = {
        "answer_wrong": t["label"] != "correct",
        "keep_or_drop": "keep" if t["label"] != "correct" else "drop",
    }
    return {"state": build_state(t), "questions": QUESTIONS, "expected": expected}

def is_wrong(t):
    return t["label"] != "correct"

def split_traces(rows):
    """Split by question, so natural and seeded runs of one question stay on the same side."""
    if all(t.get("split") is not None for t in rows):
        unknown = {t["split"] for t in rows} - {"train", "heldout"}
        assert not unknown, f"unknown split values: {sorted(unknown)}"
        method = "dataset_split_field"
        train = [t for t in rows if t["split"] == "train"]
        heldout = [t for t in rows if t["split"] == "heldout"]
    elif all(t.get("qid") is not None for t in rows):
        method = f"group_by_qid(heldout_frac={HELDOUT_FRAC}, seed={SEED})"
        qids = sorted({t["qid"] for t in rows})
        random.Random(SEED).shuffle(qids)
        heldout_qids = set(qids[:round(len(qids) * HELDOUT_FRAC)])
        train = [t for t in rows if t["qid"] not in heldout_qids]
        heldout = [t for t in rows if t["qid"] in heldout_qids]
    else:
        raise ValueError(
            "cannot split by question: the dataset needs `qid` or `split` on every trace. "
            "Re-run observer_eval_dataset_prep.ipynb so labeled_traces.jsonl carries one of them."
        )
    random.Random(SEED).shuffle(train)
    return train, heldout, method

def question_key(t):
    return t["qid"] if t.get("qid") is not None else t["question"]

train, heldout, SPLIT_METHOD = split_traces(traces)
assert train and heldout, f"empty split: train {len(train)}, held-out {len(heldout)}"

leaked = {question_key(t) for t in train} & {question_key(t) for t in heldout}
assert not leaked, f"{len(leaked)} questions appear in both train and held-out, e.g. {sorted(leaked)[:3]}"

def write_jsonl(path, rows):
    path.write_text("\n".join(json.dumps(to_laya_record(t), ensure_ascii=False) for t in rows) + "\n")

TRAIN_JSONL, HELDOUT_JSONL = WORK_DIR / "train.jsonl", WORK_DIR / "heldout.jsonl"
write_jsonl(TRAIN_JSONL, train)
write_jsonl(HELDOUT_JSONL, heldout)

base_rate = np.mean([is_wrong(t) for t in heldout])
print(f"split method: {SPLIT_METHOD}")
for name, rows in (("train", train), ("held-out", heldout)):
    n_q = len({question_key(t) for t in rows})
    print(f"{name:<9}{len(rows):>5} traces, {n_q:>5} unique questions, {np.mean([is_wrong(t) for t in rows]):.1%} wrong or partial")

split method: dataset_split_field
train     3599 traces,  2400 unique questions, 62.0% wrong or partial
held-out   900 traces,   600 unique questions, 60.0% wrong or partial


### Calibration and test portions

The held-out questions are split once more, again by question: one half fits the temperatures, the other half is the test portion. Every metric below is reported on the test portion only, so no number is measured on data that tuned it.

In [23]:
def split_by_question(rows, frac, seed):
    keys = sorted({question_key(t) for t in rows})
    random.Random(seed).shuffle(keys)
    first = set(keys[:round(len(keys) * frac)])
    return [t for t in rows if question_key(t) in first], [t for t in rows if question_key(t) not in first]

calib, test = split_by_question(heldout, CALIB_FRAC, SEED)
assert not {question_key(t) for t in calib} & {question_key(t) for t in test}, "a question is in both calibration and test"

y_calib = np.array([is_wrong(t) for t in calib])
y_test  = np.array([is_wrong(t) for t in test])
for name, rows in (("calibration", calib), ("test", test)):
    n_q = len({question_key(t) for t in rows})
    print(f"{name:<12}{len(rows):>5} traces, {n_q:>5} unique questions, {np.mean([is_wrong(t) for t in rows]):.1%} wrong or partial")

calibration   444 traces,   300 unique questions, 62.6% wrong or partial
test          456 traces,   300 unique questions, 57.5% wrong or partial


### Token window

Laya reads `max_len` tokens per question: the question and its options first, then as much of the state as fits. A longer state loses its end, and our slice ends with the final answer, so a cut slice would be judged without the thing being judged.

This cell sets `MAX_LEN` so that every slice fits, with room for the longest question, and stops if more than `MAX_TRUNCATED_FRAC` of slices would be cut. The same value goes to training and into every scoring request. The fine-tuned checkpoint stores it, so it applies by default when that checkpoint is served.

In [24]:
HEAD_TOKENS = """
import json, sys
from transformers import AutoTokenizer
from laya.common import build_head
from laya.train import to_internal

tok = AutoTokenizer.from_pretrained(sys.argv[1] + "/tokenizer")
questions = json.load(sys.stdin)
print(json.dumps({qid: len(build_head(tok, to_internal(qid, q), int(sys.argv[2]))[0]) for qid, q in questions.items()}))
"""

head_tokens = laya_python(HEAD_TOKENS, BASE_DIR, HEAD_MAX_LEN, stdin=json.dumps(QUESTIONS))
needed = lengths + max(head_tokens.values()) + 1   # question and options, the state, and a closing separator

if MAX_LEN is None:
    MAX_LEN = int(math.ceil(needed.max() / 128) * 128)
truncated = needed > MAX_LEN

print(f"question tokens: {head_tokens}")
print(f"tokens needed per slice: median {int(np.median(needed))}, p99 {int(np.percentile(needed, 99))}, max {needed.max()}")
print(f"token window: MAX_LEN = {MAX_LEN} (the base checkpoint's default is {BASE_MAX_LEN})")
print(f"would cut {int(truncated.sum())} of {len(traces)} slices ({truncated.mean():.2%})")
assert truncated.mean() <= MAX_TRUNCATED_FRAC, (
    f"{truncated.mean():.2%} of slices would lose their final answer (limit {MAX_TRUNCATED_FRAC:.0%}); raise MAX_LEN"
)

question tokens: {'answer_wrong': 41, 'keep_or_drop': 67}
tokens needed per slice: median 449, p99 807, max 1063
token window: MAX_LEN = 1152 (the base checkpoint's default is 1024)
would cut 0 of 4499 slices (0.00%)


## 4. Scoring helpers and sampler metrics

One request per trace returns both decision signals: `p(answer_wrong)` from the noul, and `p(keep)` from the choice probabilities. Laya's server answers in the same shape as Kev's.

Every request names the checkpoint it wants and carries `MAX_LEN`. The server reports whether it had to cut the state, and `score_traces` stops if it did.

**Normalized recall** = (sampler − random) / (oracle − random), at a fixed sample budget.
- sampler: share of wrong answers that land in the top-k by the signal
- random: expected share caught by random sampling (= the sample rate)
- oracle: best possible with gold labels = min(1, k / number of wrong answers)

In [25]:
# how to read each decision signal out of a Laya answer; higher = more worth a human look
SIGNAL_READ = {
    "answer_wrong": lambda a: a["noul"],
    "keep_or_drop": lambda a: a["probabilities"]["keep"],
}
assert DECISION_SIGNAL in SIGNAL_READ, f"DECISION_SIGNAL must be one of {list(SIGNAL_READ)}"

def score_traces(url, model, rows, timeout=60):
    signals, latencies, cut = {name: [] for name in SIGNAL_READ}, [], 0
    for t in rows:
        body = {"model": model, "state": build_state(t), "questions": QUESTIONS, "max_len": MAX_LEN}
        t0 = time.perf_counter()
        r = requests.post(url, json=body, timeout=timeout)
        r.raise_for_status()
        latencies.append((time.perf_counter() - t0) * 1000)
        out = r.json()
        served = (out.get("routing") or {}).get("model")
        assert served == model, f"asked for checkpoint {model!r} but {served!r} answered; check the server's LAYA_EXTRA_MODELS"
        cut += bool(out["usage"].get("truncated"))
        for name, read in SIGNAL_READ.items():
            signals[name].append(read(out["answers"][name]))
    assert cut == 0, f"the server cut {cut} of {len(rows)} states; MAX_LEN = {MAX_LEN} is too small"
    return {name: np.array(p) for name, p in signals.items()}, np.array(latencies)

def serve_command(checkpoint, name, url):
    """The shell command that serves one checkpoint under `name` on the port in `url`."""
    port = url.split(":")[2].split("/")[0]
    extra = json.dumps({name: str(checkpoint)})
    return (f"cd {LAYA_PATH} && LAYA_PORT={port} LAYA_HOST=127.0.0.1 LAYA_DEVICE={DEVICE} "
            f"LAYA_EXTRA_MODELS='{extra}' LAYA_MODELS={name} LAYA_DEFAULT_MODEL={name} "
            f"uv run --extra serve laya-serve")

def latency_summary(latencies):
    return {"p50": float(np.percentile(latencies, 50)), "p95": float(np.percentile(latencies, 95))}

def sampler_metrics(p_wrong, y_wrong, rate=SAMPLE_RATE):
    y = np.asarray(y_wrong, dtype=bool)
    n, n_wrong = len(y), int(y.sum())
    k = max(1, math.ceil(rate * n))
    top_k = np.argsort(-np.asarray(p_wrong), kind="stable")[:k]   # ties keep the caller's (shuffled) order
    sampler = y[top_k].sum() / n_wrong
    random_r = k / n
    oracle = min(1.0, k / n_wrong)
    return {
        "recall_at_rate": sampler,
        "random_recall": random_r,
        "oracle_recall": oracle,
        "normalized_recall": (sampler - random_r) / (oracle - random_r),
    }

### Realistic base rates

The test portion is about 60% wrong or partial, far above production failure rates, which makes a 5% budget too easy to fill. Sampling metrics are therefore measured on rebalanced test sets: every correct trace is kept and the wrong ones are downsampled to each target base rate. This is repeated with `N_RESAMPLES` seeds and reported as mean and standard deviation. Every model is scored on the same rebalanced sets.

AUROC does not depend on the base rate, so it is computed once on the full test portion.

In [26]:
def rebalanced_indices(y, base_rate, seed):
    """Every correct trace plus a seeded sample of wrong ones, in shuffled order, so wrong traces make up `base_rate`."""
    rng = np.random.default_rng(seed)
    correct_idx, wrong_idx = np.flatnonzero(~y), np.flatnonzero(y)
    n_wrong = round(base_rate * len(correct_idx) / (1 - base_rate))
    assert 0 < n_wrong <= len(wrong_idx), f"cannot reach base rate {base_rate} with {len(wrong_idx)} wrong traces"
    return rng.permutation(np.concatenate([correct_idx, rng.choice(wrong_idx, n_wrong, replace=False)]))

RESAMPLES = {
    br: [rebalanced_indices(y_test, br, SEED + i) for i in range(N_RESAMPLES)]
    for br in TARGET_BASE_RATES
}
RESULTS = {}   # (model, signal) -> rows of the results table

def evaluate_signal(model, signal, p):
    """Add one model's rows to RESULTS: every base rate x sample rate, mean and std over the rebalanced test sets."""
    p = np.asarray(p, dtype=float)
    auroc = roc_auc_score(y_test, p)
    rows = []
    for br, resamples in RESAMPLES.items():
        for rate in SAMPLE_RATES:
            runs = [sampler_metrics(p[idx], y_test[idx], rate) for idx in resamples]
            col = lambda key: np.array([m[key] for m in runs])
            rows.append({
                "model": model, "signal": signal, "base_rate": br, "sample_rate": rate,
                "n_rebalanced": len(resamples[0]), "auroc_full_test": auroc,
                "recall_mean": col("recall_at_rate").mean(), "recall_std": col("recall_at_rate").std(),
                "oracle_recall": col("oracle_recall").mean(), "random_recall": col("random_recall").mean(),
                "norm_recall_mean": col("normalized_recall").mean(), "norm_recall_std": col("normalized_recall").std(),
            })
    RESULTS[(model, signal)] = rows
    return rows

def results_table(models=None):
    rows = [r for key, group in RESULTS.items() if models is None or key[0] in models for r in group]
    return pd.DataFrame(rows).sort_values(["base_rate", "sample_rate", "model", "signal"], kind="stable").reset_index(drop=True)

def headline(model, signal):
    """Normalized recall at the headline SAMPLE_RATE, per base rate."""
    return {r["base_rate"]: r["norm_recall_mean"] for r in RESULTS[(model, signal)] if r["sample_rate"] == SAMPLE_RATE}

def show(table):
    print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

for br, resamples in RESAMPLES.items():
    idx = resamples[0]
    print(f"base rate {br:.0%}: {len(idx)} traces per rebalanced test set, {int(y_test[idx].sum())} wrong, {N_RESAMPLES} seeds")

base rate 10%: 216 traces per rebalanced test set, 22 wrong, 20 seeds
base rate 20%: 242 traces per rebalanced test set, 48 wrong, 20 seeds


## 5. Zero-shot baseline

The released checkpoint on the test portion, with no fine-tuning. The cell below prints the command that serves it on port 8011. Run that in a terminal, wait for it to finish loading, then run the scoring cell.

In [27]:
print(serve_command(BASE_DIR, ZERO_SHOT_MODEL, ZERO_SHOT_URL))

cd /Users/samuelabolo/kode/hobby/laya && LAYA_PORT=8011 LAYA_HOST=127.0.0.1 LAYA_DEVICE=mps LAYA_EXTRA_MODELS='{"observer-base": "/Users/samuelabolo/.cache/huggingface/hub/models--convaiinnovations--laya-multilingual/snapshots/1720e3e3357cfe1e281542e223f8273b0890ca34"}' LAYA_MODELS=observer-base LAYA_DEFAULT_MODEL=observer-base uv run --extra serve laya-serve


In [28]:
## Establish baseline with zero-shot run
p_zero, lat_zero = score_traces(ZERO_SHOT_URL, ZERO_SHOT_MODEL, test)
for signal, p in p_zero.items():
    evaluate_signal("zero_shot", signal, p)

LATENCY = {"zero_shot": latency_summary(lat_zero)}
print(f"AUROC is on the full test portion ({len(test)} traces, {y_test.mean():.1%} wrong), not on the rebalanced sets")
show(results_table(["zero_shot"]))
print(f"latency p50 {LATENCY['zero_shot']['p50']:.0f} ms, p95 {LATENCY['zero_shot']['p95']:.0f} ms")

AUROC is on the full test portion (456 traces, 57.5% wrong), not on the rebalanced sets
    model       signal  base_rate  sample_rate  n_rebalanced  auroc_full_test  recall_mean  recall_std  oracle_recall  random_recall  norm_recall_mean  norm_recall_std
zero_shot answer_wrong      0.100        0.050           216            0.446        0.045       0.035          0.500          0.051            -0.012            0.078
zero_shot keep_or_drop      0.100        0.050           216            0.459        0.027       0.036          0.500          0.051            -0.053            0.081
zero_shot answer_wrong      0.100        0.100           216            0.446        0.100       0.051          1.000          0.102            -0.002            0.057
zero_shot keep_or_drop      0.100        0.100           216            0.459        0.105       0.058          1.000          0.102             0.003            0.064
zero_shot answer_wrong      0.100        0.200           216            

## 6. Fine-tune from the released checkpoint

Laya's trainer (`laya.train.finetune`, the code behind `laya-train`) updates the whole model: the encoder at `ENCODER_LR` and the decision head at `HEAD_LR`. Before training it sets aside a tenth of the training items (at most 400) and fits its own calibration temperatures on them afterwards, so the saved checkpoint already answers with calibrated probabilities.

It is called from Python rather than through `laya-train` because the command line has no switch for gradient checkpointing, which keeps memory in check on MPS. At the settings in the config cell a run takes roughly 45 minutes on an M3 Max.

Stop the zero-shot server first if memory is tight. One training run per execution of this notebook. The log is kept in `train.log`, and the number of items Laya skipped is read back from it.

In [29]:
TRAIN_LOG, TRAIN_STATS_PATH = WORK_DIR / "train.log", WORK_DIR / "train_stats.json"
# laya-train has no switch for gradient checkpointing, so call its trainer from Python with the same settings.
TRAIN_SCRIPT = """
import json, sys
from laya.train import TrainConfig, finetune

a = json.loads(sys.argv[1])
config = TrainConfig(
    epochs=a["epochs"], micro_batch=a["micro_batch"], grad_accum=a["grad_accum"],
    encoder_lr=a["encoder_lr"], head_lr=a["head_lr"], loss=a["loss"], max_len=a["max_len"], seed=a["seed"],
    gradient_checkpointing=a["gradient_checkpointing"], amp=False, log_every=200,
)
finetune(data=a["data"], model_dir=a["base"], output_dir=a["out"], config=config, device=a["device"])
"""
train_args = {
    "data": str(TRAIN_JSONL), "base": str(BASE_DIR), "out": str(RUN_DIR),
    "epochs": TRAIN_EPOCHS, "micro_batch": TRAIN_BATCH, "grad_accum": TRAIN_ACCUM,
    "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "loss": TRAIN_LOSS, "max_len": MAX_LEN, "seed": SEED,
    "gradient_checkpointing": TRAIN_CHECKPOINTING, "device": DEVICE,
}
train_cmd = ["uv", "run", "python", "-c", TRAIN_SCRIPT, json.dumps(train_args)]
print(json.dumps(train_args, indent=1))

def run_logged(cmd, cwd, env, log_path):
    """Run a command, echo its output line by line and keep a copy in log_path."""
    with open(log_path, "w") as log, subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path.read_text()

def training_stats(log_text):
    """Item counts from Laya's own summary line. An item is one question on one trace."""
    m = re.search(r"train items (\d+), calibration items (\d+).*?skipped (\{.*?\})", log_text)
    assert m, "laya-train did not print its item summary; see train.log"
    skipped = sum(int(n) for n in re.findall(r": (\d+)", m.group(3)))
    return {"train_items": int(m.group(1)), "laya_calibration_items": int(m.group(2)), "skipped_items": skipped}

# A previous run is moved aside, never overwritten in place.
if RUN_DIR.exists():
    stamp = time.strftime("%Y%m%d-%H%M%S", time.localtime(RUN_DIR.stat().st_mtime))
    previous = RUN_DIR.with_name(f"{RUN_DIR.name}-{stamp}")
    RUN_DIR.rename(previous)
    print(f"moved the previous run to {previous}")
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
train_log = run_logged(train_cmd, LAYA_PATH, LAYA_ENV, TRAIN_LOG)
minutes = (time.time() - t0) / 60
print(f"training took {minutes:.1f} min")
assert (RUN_DIR / "model.safetensors").exists(), "laya-train finished without writing a checkpoint; see train.log"

TRAIN_STATS = {
    **training_stats(train_log), "records": len(train), "items_expected": len(train) * len(QUESTIONS),
    "epochs": TRAIN_EPOCHS, "micro_batch": TRAIN_BATCH, "grad_accum": TRAIN_ACCUM,
    "gradient_checkpointing": TRAIN_CHECKPOINTING, "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "loss": TRAIN_LOSS,
    "max_len": MAX_LEN, "minutes": round(minutes, 1), "seed": SEED,
}
TRAIN_STATS_PATH.write_text(json.dumps(TRAIN_STATS, indent=2))
print(f"Laya trained on {TRAIN_STATS['train_items']} items, kept {TRAIN_STATS['laya_calibration_items']} for its own "
      f"calibration and skipped {TRAIN_STATS['skipped_items']} (of {TRAIN_STATS['items_expected']})")
assert TRAIN_STATS["skipped_items"] == 0, "Laya skipped training items; see the reasons in train.log"

{
 "data": "/Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-laya-ft/train.jsonl",
 "base": "/Users/samuelabolo/.cache/huggingface/hub/models--convaiinnovations--laya-multilingual/snapshots/1720e3e3357cfe1e281542e223f8273b0890ca34",
 "out": "/Users/samuelabolo/kode/hobby/laya/runs/observer-laya-ft",
 "epochs": 2,
 "micro_batch": 2,
 "grad_accum": 32,
 "encoder_lr": 2.5e-05,
 "head_lr": 0.0001,
 "loss": "rlcd",
 "max_len": 1152,
 "seed": 7,
 "gradient_checkpointing": true,
 "device": "mps"
}
train items 6798, calibration items 400, eval items 400 (calibration_slice), skipped {}, device mps
optimizer budget: 6798 train item(s), effective batch up to 64 (micro_batch 2 x grad_accum 32), 214 optimizer updates over 2 epoch(s)
epoch 1/2 step 200 loss 2.1220
epoch 1/2 step 400 loss 0.4014
epoch 1/2 step 600 loss 0.7808
epoch 1/2 step 800 loss -0.0762
epoch 1/2 step 1000 loss 0.2236
epoch 1/2 step 1200 loss 0.5231
epoch 1/2 step 1400 loss 0.7279
epoch 1/2 step 1600 los

## 7. Fine-tuned model as a sampler

The cell below prints the command that serves the new run on port 8012. Run it in a terminal (stop an older server on that port first), then run the scoring cell.

Scores the calibration and test portions. Calibration scores are only used to fit temperatures in the next section; the table and the latency are from the test portion.

In [32]:
print(serve_command(RUN_DIR, FINETUNED_MODEL, FINETUNED_URL))

cd /Users/samuelabolo/kode/hobby/laya && LAYA_PORT=8012 LAYA_HOST=127.0.0.1 LAYA_DEVICE=mps LAYA_EXTRA_MODELS='{"observer-ft": "/Users/samuelabolo/kode/hobby/laya/runs/observer-laya-ft"}' LAYA_MODELS=observer-ft LAYA_DEFAULT_MODEL=observer-ft uv run --extra serve laya-serve


In [33]:
p_ft_calib, _  = score_traces(FINETUNED_URL, FINETUNED_MODEL, calib)
p_ft, lat_ft   = score_traces(FINETUNED_URL, FINETUNED_MODEL, test)
for signal, p in p_ft.items():
    evaluate_signal("finetuned", signal, p)

LATENCY["finetuned"] = latency_summary(lat_ft)
print(f"AUROC is on the full test portion ({len(test)} traces, {y_test.mean():.1%} wrong), not on the rebalanced sets")
show(results_table(["zero_shot", "finetuned"]))
for model, lat in LATENCY.items():
    print(f"{model:<10} latency p50 {lat['p50']:.0f} ms, p95 {lat['p95']:.0f} ms")

AUROC is on the full test portion (456 traces, 57.5% wrong), not on the rebalanced sets
    model       signal  base_rate  sample_rate  n_rebalanced  auroc_full_test  recall_mean  recall_std  oracle_recall  random_recall  norm_recall_mean  norm_recall_std
finetuned answer_wrong      0.100        0.050           216            0.708        0.207       0.057          0.500          0.051             0.347            0.126
finetuned keep_or_drop      0.100        0.050           216            0.723        0.286       0.068          0.500          0.051             0.524            0.150
zero_shot answer_wrong      0.100        0.050           216            0.446        0.045       0.035          0.500          0.051            -0.012            0.078
zero_shot keep_or_drop      0.100        0.050           216            0.459        0.027       0.036          0.500          0.051            -0.053            0.081
finetuned answer_wrong      0.100        0.100           216            

**Reading the result**
- AUROC near 0.5 means the scorer ranks no better than chance
- Normalized recall of 0 is random sampling, 1 is the oracle, and below 0 is worse than random
- If fine-tuned normalized recall isn't clearly above zero-shot on the test portion, ship zero-shot plus calibration and skip fine-tuning
- A rebalanced test set holds only a few dozen wrong traces, so read the standard deviation before the mean. Re-run with another `SEED` before concluding.

## 8. Temperature per decision signal

Ranking (AUROC, recall) doesn't change with temperature, but the probabilities do. The Go engine needs calibrated probabilities for the uncertainty bonus and alert thresholds.

Each temperature is fitted on the calibration portion only. ECE is reported on the test portion, before and after scaling, so it measures calibration on data the temperature has not seen.

This temperature sits on top of the one Laya fitted itself during training. A value near 1 means Laya's own calibration already holds on these traces.

In [34]:
def fit_temperature(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    logits = np.log(p / (1 - p))
    def nll(T):
        q = 1 / (1 + np.exp(-logits / T))
        q = np.clip(q, 1e-9, 1 - 1e-9)
        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x

def ece(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum(abs(p[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
               for b in range(bins) if (idx == b).any())

def apply_temperature(p, T):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return 1 / (1 + np.exp(-np.log(p / (1 - p)) / T))

TEMPERATURES, ECE = {}, {}
for signal in SIGNAL_READ:
    T = fit_temperature(p_ft_calib[signal], y_calib.astype(float))
    TEMPERATURES[signal] = round(float(T), 4)
    ECE[signal] = {
        "before": float(ece(p_ft[signal], y_test)),
        "after": float(ece(apply_temperature(p_ft[signal], T), y_test)),
    }
    print(f"{signal:<14} temperature {T:.2f} (fitted on {len(calib)} calibration traces); "
          f"test ECE before {ECE[signal]['before']:.3f}, after {ECE[signal]['after']:.3f}")
    if not 0.06 < T < 19.9:
        print(f"  WARNING: {signal} temperature hit the search bound, so the signal is badly miscalibrated. "
              f"Check that the served run was trained on this question.")

answer_wrong   temperature 5.00 (fitted on 444 calibration traces); test ECE before 0.294, after 0.043
keep_or_drop   temperature 4.59 (fitted on 444 calibration traces); test ECE before 0.287, after 0.096


## 9. Heuristic baseline (Arm D)

Scores the test portion with simple rules and no model. If a few `if` statements find wrong answers as well as fine-tuned Laya does, the model isn't earning its latency.

Each rule is oriented so that a higher value is more suspicious:
- `short_answer`: fewer words in the final answer
- `few_evidence`: fewer evidence items
- `no_search`: `used_search` is false
- `tool_error`: any tool call with a status other than `ok`
- `bad_stop`: `stop_reason` other than `answered`
- `low_overlap`: share of the final answer's words that do not appear in the evidence text

The rules are combined with a logistic regression fitted on the train split, and each rule is also scored on its own.

In [35]:
def words(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def heuristic_features(t):
    answer_words = words(t["final_answer"])
    evidence_words = set(words(" ".join(t.get("evidence", []))))
    overlap = np.mean([w in evidence_words for w in answer_words]) if answer_words else 0.0
    tool_calls = t.get("tool_calls", [])
    return {
        "short_answer": -len(answer_words),
        "few_evidence": -len(t.get("evidence", [])),
        "no_search":    float(not t.get("used_search", any(c.get("name") == "search" for c in tool_calls))),
        "tool_error":   float(any(c.get("status", "ok") != "ok" for c in tool_calls)),
        "bad_stop":     float(t.get("stop_reason", "answered") != "answered"),
        "low_overlap":  1.0 - overlap,
    }

RULES = list(heuristic_features(train[0]))
X_train = np.array([[heuristic_features(t)[r] for r in RULES] for t in train], dtype=float)
X_test  = np.array([[heuristic_features(t)[r] for r in RULES] for t in test], dtype=float)
y_train = np.array([is_wrong(t) for t in train])

print(f"{'rule':<14}{'mean when correct':>20}{'mean when wrong':>18}   (test portion)")
for j, rule in enumerate(RULES):
    print(f"{rule:<14}{X_test[~y_test, j].mean():>20.3f}{X_test[y_test, j].mean():>18.3f}")

rule             mean when correct   mean when wrong   (test portion)
short_answer                -2.046            -6.412
few_evidence                -3.237            -2.969
no_search                    0.005             0.076
tool_error                   0.010             0.053
bad_stop                     0.000             0.004
low_overlap                  0.155             0.254


In [36]:
heuristic_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
heuristic_model.fit(X_train, y_train)
evaluate_signal("heuristic", "logreg", heuristic_model.predict_proba(X_test)[:, 1])
for j, rule in enumerate(RULES):
    evaluate_signal("heuristic", rule, X_test[:, j])

weights = dict(zip(RULES, heuristic_model[-1].coef_[0].round(3).tolist()))
print("logistic regression weights (standardized features):", weights)
print(f"AUROC is on the full test portion ({len(test)} traces, {y_test.mean():.1%} wrong), not on the rebalanced sets")
show(results_table(["heuristic"]).query("sample_rate == @SAMPLE_RATE"))

logistic regression weights (standardized features): {'short_answer': -1.82, 'few_evidence': 0.259, 'no_search': -0.009, 'tool_error': 0.058, 'bad_stop': 0.392, 'low_overlap': 0.277}
AUROC is on the full test portion (456 traces, 57.5% wrong), not on the rebalanced sets
    model       signal  base_rate  sample_rate  n_rebalanced  auroc_full_test  recall_mean  recall_std  oracle_recall  random_recall  norm_recall_mean  norm_recall_std
heuristic     bad_stop      0.100        0.050           216            0.502        0.050       0.052          0.500          0.051            -0.002            0.115
heuristic few_evidence      0.100        0.050           216            0.544        0.143       0.061          0.500          0.051             0.205            0.137
heuristic       logreg      0.100        0.050           216            0.701        0.314       0.072          0.500          0.051             0.585            0.160
heuristic  low_overlap      0.100        0.050           

### Success bar

The project's bar: the fine-tuned model, here Laya, read through `DECISION_SIGNAL`, beats the best heuristic by at least `SUCCESS_MARGIN` (10 points) of normalized recall at the 5% budget, at every target base rate.

The best heuristic and the best single rule are picked on the test portion, per base rate. That choice favours the heuristic, so a pass here is a conservative one.

In [37]:
laya_recall = headline("finetuned", DECISION_SIGNAL)
SUCCESS_BAR, HEURISTIC_BEST = {}, {}
for br in TARGET_BASE_RATES:
    by_rule = {rule: headline("heuristic", rule)[br] for rule in RULES}
    best_rule = max(by_rule, key=by_rule.get)
    candidates = {"logreg": headline("heuristic", "logreg")[br], best_rule: by_rule[best_rule]}
    best = max(candidates, key=candidates.get)
    margin = laya_recall[br] - candidates[best]
    HEURISTIC_BEST[str(br)] = {
        "best_single_rule": best_rule, "best_single_rule_norm_recall": by_rule[best_rule],
        "logreg_norm_recall": candidates["logreg"], "best_heuristic": best, "best_heuristic_norm_recall": candidates[best],
    }
    SUCCESS_BAR[str(br)] = {"finetuned_norm_recall": laya_recall[br], "margin": margin, "passed": bool(margin >= SUCCESS_MARGIN)}
    print(f"base rate {br:.0%}, normalized recall at {SAMPLE_RATE:.0%}: fine-tuned Laya ({DECISION_SIGNAL}) {laya_recall[br]:.3f}, "
          f"logreg {candidates['logreg']:.3f}, best single rule {best_rule} {by_rule[best_rule]:.3f} "
          f"-> margin over best heuristic ({best}) {margin:+.3f}: {'PASS' if margin >= SUCCESS_MARGIN else 'FAIL'}")

verdict = "meets" if all(s["passed"] for s in SUCCESS_BAR.values()) else "does NOT meet"
print(f"\nFine-tuned Laya {verdict} the success bar of +{SUCCESS_MARGIN:.2f} normalized recall over the best heuristic at every base rate.")

base rate 10%, normalized recall at 5%: fine-tuned Laya (keep_or_drop) 0.524, logreg 0.585, best single rule few_evidence 0.205 -> margin over best heuristic (logreg) -0.061: FAIL
base rate 20%, normalized recall at 5%: fine-tuned Laya (keep_or_drop) 0.784, logreg 0.813, best single rule few_evidence 0.314 -> margin over best heuristic (logreg) -0.029: FAIL

Fine-tuned Laya does NOT meet the success bar of +0.10 normalized recall over the best heuristic at every base rate.


## 10. Results table

One row per model, signal, base rate and sample rate. `recall` and `norm_recall` are mean and standard deviation over the `N_RESAMPLES` rebalanced test sets; `auroc_full_test` is from the full test portion and repeats across a model's rows.

In [38]:
results = results_table()
results.to_csv(WORK_DIR / "results.csv", index=False)
print(f"{len(results)} rows, saved to {WORK_DIR / 'results.csv'}")
print(f"AUROC is on the full test portion ({len(test)} traces, {y_test.mean():.1%} wrong), not on the rebalanced sets")
results.style.format(precision=3).hide(axis="index")

66 rows, saved to /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-laya-ft/results.csv
AUROC is on the full test portion (456 traces, 57.5% wrong), not on the rebalanced sets


model,signal,base_rate,sample_rate,n_rebalanced,auroc_full_test,recall_mean,recall_std,oracle_recall,random_recall,norm_recall_mean,norm_recall_std
finetuned,answer_wrong,0.100,0.050,216,0.708,0.207,0.057,0.500,0.051,0.347,0.126
finetuned,keep_or_drop,0.100,0.050,216,0.723,0.286,0.068,0.500,0.051,0.524,0.150
heuristic,bad_stop,0.100,0.050,216,0.502,0.050,0.052,0.500,0.051,-0.002,0.115
heuristic,few_evidence,0.100,0.050,216,0.544,0.143,0.061,0.500,0.051,0.205,0.137
heuristic,logreg,0.100,0.050,216,0.701,0.314,0.072,0.500,0.051,0.585,0.160
heuristic,low_overlap,0.100,0.050,216,0.613,0.052,0.046,0.500,0.051,0.003,0.103
heuristic,no_search,0.100,0.050,216,0.536,0.125,0.075,0.500,0.051,0.165,0.166
heuristic,short_answer,0.100,0.050,216,0.357,0.036,0.040,0.500,0.051,-0.032,0.088
heuristic,tool_error,0.100,0.050,216,0.522,0.084,0.054,0.500,0.051,0.074,0.121
zero_shot,answer_wrong,0.100,0.050,216,0.446,0.045,0.035,0.500,0.051,-0.012,0.078


## 11. Model manifest

Records which run this is, how to calibrate it and what quality to expect, in the same shape as the Kev notebook's manifest.

`decision` names the question to decide on and how to read it: a `noul` answer carries the probability in `noul`, a `choice` answer carries it in `probabilities[positive_option]`. `serving` holds what a client needs to reach this checkpoint: the name it is registered under and the token window.

In [39]:
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

def n_questions(rows):
    return len({question_key(t) for t in rows})

decision_type = QUESTIONS[DECISION_SIGNAL]["type"]
train_stats = json.loads(TRAIN_STATS_PATH.read_text())   # written by the training cell

manifest = {
    "model": "observer-laya-ft",
    "family": "laya",
    "run_dir": str(RUN_DIR),
    "base": BASE_MODEL,
    "base_dir": str(BASE_DIR),
    "serving": {"model": FINETUNED_MODEL, "max_len": MAX_LEN, "command": serve_command(RUN_DIR, FINETUNED_MODEL, FINETUNED_URL)},
    "train_data_sha": file_hash(TRAIN_JSONL),
    "seed": SEED,
    "questions": QUESTIONS,
    "temperatures": TEMPERATURES,
    "decision": {
        "question": DECISION_SIGNAL,
        "type": decision_type,
        "probability_field": "noul" if decision_type == "noul" else "probabilities",
        "positive_option": "keep" if decision_type == "choice" else None,
        "temperature": TEMPERATURES[DECISION_SIGNAL],
    },
    "split": {
        "method": SPLIT_METHOD,
        "train": {"traces": len(train), "questions": n_questions(train)},
        "calibration": {"traces": len(calib), "questions": n_questions(calib), "base_rate": round(float(y_calib.mean()), 4)},
        "test": {"traces": len(test), "questions": n_questions(test), "base_rate": round(float(y_test.mean()), 4)},
    },
    "heldout": {"n": len(heldout), "base_rate": round(float(base_rate), 4)},
    "token_limits": {
        "max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN, "base_max_len": BASE_MAX_LEN,
        "question_tokens": head_tokens, "max_state_tokens": int(lengths.max()), "slices_truncated": int(truncated.sum()),
    },
    "slice": {"max_evidence": MAX_EVIDENCE, "max_evidence_chars": MAX_EVIDENCE_CHARS},
    "training": train_stats,
    "calibration": {"fitted_on": "calibration", "n": len(calib), "temperatures": TEMPERATURES, "test_ece": ECE},
    "evaluation": {
        "target_base_rates": TARGET_BASE_RATES, "sample_rates": SAMPLE_RATES, "n_resamples": N_RESAMPLES,
        "auroc_on": "full test portion", "sampling_metrics_on": "rebalanced test sets",
    },
    "metrics": {
        model: {
            "auroc": RESULTS[(model, DECISION_SIGNAL)][0]["auroc_full_test"],
            "normalized_recall": {str(br): v for br, v in headline(model, DECISION_SIGNAL).items()},
            "sample_rate": SAMPLE_RATE,
            "signal": DECISION_SIGNAL,
        }
        for model in ("zero_shot", "finetuned")
    },
    "latency_ms": {**LATENCY["finetuned"], **LATENCY},   # top-level p50 / p95 are the fine-tuned model's
    "results": results.to_dict("records"),
    "heuristic_baseline": {
        "rules": RULES,
        "logreg_weights": weights,
        "best": HEURISTIC_BEST,
        "results": results[results.model == "heuristic"].to_dict("records"),
    },
    "success_bar": {"margin_required": SUCCESS_MARGIN, "sample_rate": SAMPLE_RATE, "by_base_rate": SUCCESS_BAR},
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print(f"wrote {WORK_DIR / 'manifest.json'}")
print(f"decision question: {DECISION_SIGNAL} ({decision_type}), temperature {TEMPERATURES[DECISION_SIGNAL]}")

wrote /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-laya-ft/manifest.json
decision question: keep_or_drop (choice), temperature 4.5891
